In [22]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from preprocessing import (RANDOM_STATE, columns_type,fill_missing, fold_matrices, held_out_ap,
                           load_data, paired_delta, target, temporal_splits)
from collections import Counter
from sklearn.feature_selection import SelectKBest, f_classif
import warnings
from collections import namedtuple
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pointbiserialr



train_clientes = load_data()
numeric, categorical = columns_type(train_clientes)
frame = train_clientes[numeric + categorical + [target]]
time_splits = temporal_splits(frame)

# Provisional recipe: rules only, until 1.7.2 and 1.8 are closed
recipe = {
    "numeric": numeric,
    "categorical": categorical,
    "fill": [(numeric, SimpleImputer(strategy="median")),
             (categorical, SimpleImputer(strategy="most_frequent"))],
    "log1p": [],  # the outlier treatment chosen in 1.7.2 goes here
}


print(len(numeric), "numeric |", len(categorical), "categorical |", len(time_splits), "splits")

37 numeric | 12 categorical | 7 splits


Before moving on to the feature selection techniques, we need to set a **reference** for the results given by our metric (**AP** -> average precision), so that we can then compare them with the results obtained when the techniques are applied.

The function `held_out_ap` receives four parameters:

1. `frame` (the data)
2. `time_splits` (the 7 temporal splits: each one trains on the past and validates on the period right after it)
3. `recipe` (The pre-processing methods defined in the previus notebooks)
4. `select` (receives the feature selection technique to apply inside each split)

> **Note:** the reference level leaves it empty, meaning no feature selection technique was used.

After getting these, for each split it will prepare the training and validation rows with the recipe, finally trains a logistic regression on the training rows and scores it on the validation rows.

In [23]:
reference = held_out_ap(frame, time_splits, recipe)
print(reference[["n", "AP", "F1"]])
print(f"mean: AP {reference['AP'].mean():.4f} | F1 {reference['F1'].mean():.4f}")

     n        AP        F1
0  562  0.824279  0.701987
1  585  0.838774  0.699213
2  624  0.853865  0.745583
3  650  0.817163  0.722045
4  694  0.801795  0.687386
5  753  0.858422  0.708464
6  770  0.854445  0.758824
mean: AP 0.8355 | F1 0.7176


---

## 1. Feature Selection

Feature Selection is the process of selecting the most relevant features from a dataset to improve model performance. To simplify the model, this section aims to reduce the number of features, reduce computational complexity and potentially improve its generalization ability.

---

### 1.1 Filter Methods 

Filter methods are one of the feature selection techniques that evaluate the relevance of features based on statistical measures without involving a machine learning model.

Selects the top-ranked features and removes irrelevant or redundant ones before model training.

---

#### 1.1.1 Variance Thresholding

Variance Thresholding removes features with very low variance, as they contribute little useful information to the learning process. Features with almost constant values are typically discarded.

In [24]:
# Split 0's training matrix, prepared by the recipe
train_rows, validation_rows = time_splits[0]
A_train, _, names = fold_matrices(frame.iloc[train_rows], frame.iloc[validation_rows], recipe)

variances = pd.Series(A_train.var(axis=0), index=names)
measured = variances.index.isin(numeric)
print(f"{len(variances)} columns, {int((variances == 0).sum())} constant")
print(f"dummies : variance {variances[~measured].min():.5f} to {variances[~measured].max():.4f}")
print(f"measured: variance {variances[measured].min():.4f} to {variances[measured].max():.4f}")

562 columns, 0 constant
dummies : variance 0.00045 to 0.2297
measured: variance 1.0000 to 1.0000


In [25]:

variance = held_out_ap(frame, time_splits, recipe, select=lambda: VarianceThreshold(0.0))
print(variance[["n", "AP", "F1"]])
print("AP:", paired_delta(variance["AP"], reference["AP"]))
print("F1:", paired_delta(variance["F1"], reference["F1"]))

     n        AP        F1
0  562  0.824279  0.701987
1  585  0.838774  0.699213
2  624  0.853865  0.745583
3  650  0.817163  0.722045
4  694  0.801795  0.687386
5  753  0.858422  0.708464
6  770  0.854445  0.758824
AP: {'delta': np.float64(0.0), 'SEM': np.float64(0.0), 'wins': '0/7', 'verdict': 'identical on every split'}
F1: {'delta': np.float64(0.0), 'SEM': np.float64(0.0), 'wins': '0/7', 'verdict': 'identical on every split'}


**Conclusions:** The filter removed zero columns because a threshold of 0.0 only drops features with zero variance. Since no columns were constant, the dataset remained unchanged.

#### 1.1.2 Select k-Best

**Columns relevancy**:

In [26]:
def cramers_v(column, outcome):
    """Chi-square on a contingency table, rescaled so columns compare.

        Returns the statistic, its p-value, Cramer's V, and the smallest expected
    count. The p-value is an approximation that needs every expected count above
    about 5, so a level almost nobody occupies breaks it. V is still readable as
    an effect size.
    """
    table = pd.crosstab(column, outcome)
    if min(table.shape) < 2:
        return 0.0, float("nan"), 0.0, float(table.to_numpy().min())
    statistic, p, _, expected = chi2_contingency(table, correction=False)
    v = np.sqrt(statistic / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return statistic, p, v, expected.min()


def typed_relevance(rows, outcome):
    """One comparable number per SOURCE column, each by the index its type allows.

    |point-biserial r| for a measurement, Cramer's V for a categorical. Both are
    on [0, 1], so a fourteen-level column and a duration can be ranked together.
    A measurement is read as the recipe leaves it, because that is the column the
    model will see.
    """
    scores = {}
    for column in numeric:
        values = rows[column].to_numpy(dtype=float)
        if column in recipe["log1p"]:
            values = np.log1p(values)
        scores[column] = abs(
            pointbiserialr(outcome.to_numpy(dtype=int), values).statistic)
    for column in categorical:
        scores[column] = cramers_v(rows[column].astype(str), outcome)[2]
    return pd.Series(scores).sort_values(ascending=False)
     

In [27]:
# Split 0's training rows, filled by the recipe
train_rows, validation_rows = time_splits[0]
first_rows, _ = fill_missing(frame.iloc[train_rows], frame.iloc[validation_rows], recipe["fill"])
first_y = first_rows[target]

source_scores = typed_relevance(first_rows, first_y)
table = []
for column in source_scores.index:
    if column in numeric:
        table.append({"column": column, "index": "|point-biserial r|",
                      "value": source_scores[column]})
    else:
        statistic, p, v, expected = cramers_v(first_rows[column].astype(str), first_y)
        table.append({"column": column, "index": "Cramer's V", "value": v,
                      "levels": first_rows[column].nunique(), "chi2": statistic,
                      "p": p, "min expected": expected})
print(pd.DataFrame(table).round(4).to_string(index=False, na_rep=""))

                        column              index  value  levels     chi2      p  min expected
                          city         Cramer's V 0.4479   160.0 447.1810 0.0000        0.4365
                   postal_area         Cramer's V 0.3761   142.0 315.3413 0.0000        0.4365
                  municipality         Cramer's V 0.3517   119.0 275.7435 0.0000        0.4365
              transport_method         Cramer's V 0.3166    52.0 223.4599 0.0000        0.4365
            purchase_count_90d |point-biserial r| 0.3132                                      
                    spend_365d |point-biserial r| 0.3104                                      
                    spend_180d |point-biserial r| 0.3074                                      
           purchase_count_180d |point-biserial r| 0.3052                                      
           purchase_count_365d |point-biserial r| 0.2986                                      
                     spend_90d |point-biserial r| 

**Conclusions:**

In [28]:
kbest_runs = {}
for k in (20, 50, 100):
    kbest_runs[k] = held_out_ap(frame, time_splits, recipe, select=lambda: SelectKBest(f_classif, k=k))
    print(f"SelectKBest(k={k}): AP {kbest_runs[k]['AP'].mean():.4f} | F1 {kbest_runs[k]['F1'].mean():.4f}")
    for metric in ("AP", "F1"):
        r = paired_delta(kbest_runs[k][metric], reference[metric])
        print(f"   d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")

# Columns k=20 kept in every split
kept = Counter(c for columns in kbest_runs[20]["kept"] for c in columns)
print([c for c, n in kept.items() if n == len(time_splits)])

SelectKBest(k=20): AP 0.8434 | F1 0.7394
   dAP +0.0079 +/- 0.0071, better on 5/7 -> inside the noise
   dF1 +0.0217 +/- 0.0089, better on 5/7 -> better
SelectKBest(k=50): AP 0.8461 | F1 0.7339
   dAP +0.0106 +/- 0.0054, better on 5/7 -> inside the noise
   dF1 +0.0163 +/- 0.0068, better on 6/7 -> better
SelectKBest(k=100): AP 0.8413 | F1 0.7265
   dAP +0.0058 +/- 0.0027, better on 6/7 -> better
   dF1 +0.0089 +/- 0.0065, better on 5/7 -> inside the noise
[np.str_('transport_method_Maybe Tomorrow Delivery 11'), np.str_('commercial_region_Region RG23: We Will See Later'), np.str_('municipality_Municipality MU70: Council of Last Minute Edits'), np.str_('postal_area_Postal area PA84: Hidden Sheet Sector'), np.str_('prior_purchase_count'), np.str_('prior_total_spend'), np.str_('prior_std_purchase_value'), np.str_('prior_max_purchase_value'), np.str_('purchase_count_30d'), np.str_('spend_30d'), np.str_('purchase_count_90d'), np.str_('spend_90d'), np.str_('purchase_count_180d'), np.str_('spe

---

### Wrapper Methods 

### Embedded Methods 

### Feature Extraction

## Dimensionality Reduction 